# Historical feature and classifier comparison

Curated from the original collaborative PBL repository (source blob `d01fe676fb312f132ab69b4021ae50dcf3568f33`). Original cell order and research logic are retained; local PBL paths were made relative and saved outputs/execution metadata were cleared for privacy and size. This archival notebook documents exploratory feature/classifier experiments, not a freshly validated pipeline. Original datasets and embeddings are not included. Execute individual cells only after supplying their inputs and reviewing alternative branches. Some original cells use earlier APIs or write experiment outputs; see [data preparation notes](../../docs/data_preparation.md).


In [ ]:
import sys

sys.path.append('..')

In [ ]:
import numpy as np
from src.preprocessing import batch_encode_and_flatten
import pandas as pd
from pathlib import Path
from src.utils import calculate_search_grid

# Import dataset / sequences
train_path = Path('../data/split/exp_proteins_c30_flens2-98_enz_train.tsv')
test_path = Path('../data/split/exp_proteins_c30_flens2-98_enz_test.tsv')

df_train = pd.read_csv(train_path, sep='\t')
df_test = pd.read_csv(test_path, sep='\t')

In [ ]:
from protlearn import features
# Get and encode the sequences using AA composition
train_list = list(df_train['Sequence'])
test_list = list(df_test['Sequence'])

# One-hot encode the sequences
X_train = features.aaindex1(train_list, standardize='zscore')[0]
X_test = features.aaindex1(test_list, standardize='zscore')[0]

# Define labels
y_train = list(df_train['Enzyme'])
y_test = list(df_test['Enzyme'])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import LinearSVC

# Initializing the estimators
rf = RandomForestClassifier()
logr = LogisticRegression(max_iter=1000)
svm = LinearSVC()

In [ ]:
from sklearn.metrics import matthews_corrcoef

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))

In [ ]:
logr.fit(X_train, y_train)
y_pred = logr.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))

In [ ]:
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))

In [ ]:
# Random Forest
n_iter = 10
param_dist_rf = {'n_estimators': [100],'max_depth': [None, 10, 20, 30], 'max_features': ['sqrt', 'log2'], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2, 4]}
rand_grid_rf = calculate_search_grid(rf, X_train, y_train, X_test, y_test, n_iter, param_dist_rf)
rand_grid_rf.to_csv('../results/performance_eval/gridsearch_model_comparison/aaindex/aaindex_rf.tsv', sep='\t', index=False)

In [ ]:
# Logistic Regression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
n_iter = 10
param_dist_logr = {'max_iter': [1000], 'C': [0.1, 1, 10, 100, 1000], 'tol': [0.00001, 0.0001, 0.001, 0.01]}
rand_grid_logr = calculate_search_grid(logr, X_train, y_train, X_test, y_test, n_iter, param_dist_logr, randomized=False)
rand_grid_logr.to_csv('../results/performance_eval/gridsearch_model_comparison/aaindex/aaindex_logr.tsv', sep='\t', index=False)

In [ ]:
# SVM
n_iter = 10
param_dist_svm = {'max_iter': [1000], 'loss': ['hinge', 'squared_hinge'], 'C': [1, 10, 100], 'fit_intercept': [True, False]}
rand_grid_svm = calculate_search_grid(svm, X_train, y_train, X_test, y_test, n_iter, param_dist_svm, randomized=True)
rand_grid_svm.to_csv('../results/performance_eval/gridsearch_model_comparison/aaindex/aaindex_svm.tsv', sep='\t', index=False)

In [ ]:
from sklearn.model_selection import cross_val_score
logr = LinearSVC(max_iter=10000 ,C=1000, tol=0.0001)
logr.fit(X_train, y_train)
preds = logr.predict(X_test)

In [ ]:
accuracy_score(y_test, preds)
